# V3&4 — Hybrid GPU SIFT (Gaussian Pyramid + Descriptor 128 chiều)

Notebook này giải thích và chạy lại chương trình `v8_T4_cupy_hybrid_sift.py`, phiên bản **kết hợp 2 bước nặng nhất** của thuật toán SIFT (Scale-Invariant Feature Transform) và tăng tốc chúng bằng **CuPy + CUDA RawKernel** trên GPU (được tối ưu cho GPU Tesla T4 trên Google Colab):

1. **Xây dựng Gaussian Pyramid** — làm mờ ảnh nhiều lần với các độ lệch chuẩn (sigma) khác nhau, rồi thu nhỏ (downsample) để tạo ra "tháp" ảnh nhiều tỉ lệ (octave).
2. **Trích xuất Descriptor 128 chiều** cho mỗi keypoint — vector đặc trưng mô tả "hình dạng cục bộ" quanh điểm đặc trưng, dùng để so khớp ảnh (image matching).

Ý tưởng cốt lõi: **CPU (OpenCV)** vẫn đảm nhiệm bước *tìm điểm đặc trưng* (keypoint detection) vì đây không phải nút thắt cổ chai hiệu năng, còn **GPU** đảm nhiệm 2 bước tốn phép tính ma trận/convolution nặng nhất — pyramid và descriptor — để tận dụng khả năng tính toán song song hàng nghìn luồng.

Notebook được chia thành các phần:
- Cài đặt & import thư viện
- Mã nguồn CUDA C++ (3 kernel: Gaussian blur, downsample, descriptor)
- Biên dịch kernel bằng CuPy
- Các hàm Python hỗ trợ (tạo Gaussian kernel 1D, pyramid CPU, pyramid GPU)
- Hàm benchmark so sánh tốc độ CPU vs GPU
- Chạy benchmark


## 1. Cài đặt và import thư viện

- **OpenCV (`cv2`)**: đọc ảnh, các hàm SIFT chuẩn trên CPU (dùng để đối chiếu / benchmark và để lấy toạ độ keypoint).
- **CuPy (`cupy`)**: thư viện giống NumPy nhưng chạy trực tiếp trên GPU Nvidia; đồng thời cho phép biên dịch và gọi kernel CUDA C++ thô (`cp.RawKernel`).
- **Google Colab Drive**: nếu notebook đang chạy trên Colab, đoạn code sẽ tự mount Google Drive để đọc bộ ảnh test (DIV2K).

> Nếu bạn không chạy trên Colab hoặc không có Google Drive, phần mount sẽ tự bỏ qua (đã được bọc trong `try/except`).


In [9]:
import sys
import os
import time
import math
import glob
import numpy as np

# Cấu hình lại chuẩn mã hóa đầu ra (tránh lỗi font tiếng Việt nếu có)
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Kiểm tra và import thư viện OpenCV (dùng để đọc ảnh và các hàm CPU cơ bản)
try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    CV2_AVAILABLE = False

# Hỗ trợ liên kết (mount) Google Drive nếu đang chạy trên Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    pass

# Kiểm tra và import thư viện CuPy (Chạy mảng đa chiều trực tiếp trên GPU Nvidia)
try:
    import cupy as cp
    CUPY_AVAILABLE = True
except Exception as e:
    CUPY_AVAILABLE = False
    print(f"[ERROR] Không import được CuPy: {e}")

# Thêm đường dẫn thư mục làm việc vào hệ thống
folder_path = '/content/drive/MyDrive/Colab Notebooks'
if folder_path not in sys.path:
    sys.path.insert(0, folder_path)


Mounted at /content/drive


## 2. Mã nguồn CUDA C++

Phần này định nghĩa 3 kernel CUDA (hàm chạy song song trên GPU), được viết bằng chuỗi C++ (`cuda_code`) và sẽ được CuPy biên dịch ngay lúc chạy (JIT — Just In Time compilation).

### 2.1. `fused_gaussian_kernel` — Gaussian Blur tách biến (separable convolution)

Đây là kernel làm mờ ảnh theo phân bố Gauss, dùng kỹ thuật **tách chiều** (separable filter): thay vì nhân chập 2D tốn kém (O(r²) phép tính mỗi pixel), ta nhân chập **theo hàng ngang trước, rồi theo hàng dọc sau** (O(2r) phép tính) — kết quả toán học tương đương nhưng nhanh hơn nhiều.

Kỹ thuật tối ưu chính được dùng:
- **Shared Memory** (`sm_input`, `sm_row`): bộ nhớ cực nhanh dùng chung giữa các luồng trong cùng 1 block. Dữ liệu ảnh (kèm viền mở rộng theo bán kính lọc) được tải 1 lần vào shared memory, sau đó mọi luồng tính toán trên dữ liệu này thay vì đọc lại từ Global Memory (chậm hơn nhiều).
- **Reflection padding**: khi vùng lọc vượt ra ngoài biên ảnh, kernel "phản chiếu" pixel biên (giống đặt gương ở mép ảnh) thay vì đọc dữ liệu rác hoặc lỗi tràn bộ nhớ.
- **3 vòng lặp tuần tự với `__syncthreads()`**:
  1. Tải ảnh (có viền) vào shared memory.
  2. Nhân chập theo chiều ngang, lưu kết quả trung gian vào `sm_row`.
  3. Nhân chập theo chiều dọc trên `sm_row`, ghi kết quả cuối ra `d_output`.

Mỗi block xử lý 1 tile 16×16 pixel của ảnh.

### 2.2. `downsample_kernel` — Thu nhỏ ảnh còn 1 nửa

Kernel đơn giản: mỗi luồng xuất ra 1 pixel = ảnh gốc lấy tại toạ độ `(x*2, y*2)` (lấy mẫu xuống — subsampling, không nội suy) để chuyển sang octave (tỉ lệ) tiếp theo trong Gaussian Pyramid.

### 2.3. `compute_sift_descriptors` — Sinh vector đặc trưng 128 chiều

Đây là kernel phức tạp nhất. **Mỗi block GPU xử lý đúng 1 keypoint**, với 256 threads (khớp với vùng 16×16 pixel quanh điểm đó). Quy trình:

1. **Lấy vùng lân cận 16×16** quanh keypoint, xoay theo góc chủ đạo (`kpt_angle`) để đảm bảo tính bất biến khi ảnh bị xoay (rotation invariance).
2. **Tính gradient** (độ lớn `mag` và hướng `ori`) tại từng pixel bằng sai phân trung tâm (central difference).
3. **Trọng số Gaussian**: pixel càng xa tâm keypoint càng có ảnh hưởng thấp lên descriptor.
4. **Phân bin vào histogram 128 chiều**: chia vùng 16×16 thành lưới 4×4 ô vuông, mỗi ô có histogram 8 hướng gradient → 4×4×8 = 128 chiều. Dùng `atomicAdd` để tránh xung đột khi nhiều thread ghi cùng 1 ô nhớ.
5. **Chuẩn hoá 3 bước** (norm → cắt ngưỡng 0.2 → norm lại) để descriptor bất biến với thay đổi độ sáng (illumination invariance) — đúng chuẩn thuật toán SIFT gốc của David Lowe.

> Cả 3 kernel đều được biên dịch với cờ `--use_fast_math` để tăng tốc các hàm lượng giác (sin, cos, sqrt...).


In [10]:
cuda_code = r'''
// ---------------------------------------------------------
// 1. KERNELS CHO GAUSSIAN PYRAMID
// Mục đích: Áp dụng bộ lọc làm mờ ảnh (Gaussian blur) theo chiều dọc và ngang.
// ---------------------------------------------------------
extern "C" __global__
void fused_gaussian_kernel(const float* d_input, float* d_output, const float* d_kernel, 
                           int radius, int H, int W) 
{
    // Khai báo Shared Memory (Bộ nhớ chia sẻ tốc độ siêu cao trên GPU)
    // Các luồng trong cùng 1 khối (block) sẽ dùng chung bộ nhớ này.
    __shared__ float sm_input[48][48];
    __shared__ float sm_row[48][16];

    // Lấy tọa độ của luồng (thread) và khối (block) hiện tại
    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int bx = blockIdx.x;
    int by = blockIdx.y;
    
    int tid = ty * 16 + tx; // chuyển 2D thành 1D
    int x0 = bx * 16; // Gốc toạ độ, mỗi block chịu trách nhiệm 16x16, nên lấy
    int y0 = by * 16; //chỉ số block (bx, by) *16 để ra toạ độ gốc của bức ảnh
    // khi chạy Gaussian Blur hay convolution các pixel nằm ở rìa của vùng 16x16 sẽ cần đọc thông tin 
    // từ các pixel lân cận nằm ở ngoài vùng đó
    int tile_size = 16 + 2 * radius; // Chiều dài cạnh của vùng dữ liệu thực tế cần tải vào bộ nhớ chia sẻ
    int total_elements = tile_size * tile_size; // Tổng số phần tử (pixel) nằm trong khối dữ liệu mở rộng này
    
    // VÒNG LẶP 1: Tải dữ liệu ảnh từ Global Memory vào Shared Memory
    // Có xử lý ngoại lệ (reflection padding) khi vượt quá viền ảnh để không bị lỗi.
    for (int i = tid; i < total_elements; i += 256) {

        // tính toán toạ độ toàn cục và cục bộ
        // Vị trí tương đối bên trong vùng nhớ Shared Memory
        int r_y = i / tile_size; 
        int r_x = i % tile_size;
        
        // Vị trí thực tế trên bức ảnh lớn. Chú ý đoạn y0 - radius: Nó lùi tọa độ gốc lại một khoảng bằng bán kính 
        // (radius) để lấy trọn vẹn phần viền ảnh cần thiết cho phép tính làm mờ
        int gy = y0 - radius + r_y;
        int gx = x0 - radius + r_x;
        
        // Khi quét dữ liệu ở sát mép ảnh, phần viền (radius) sẽ bị văng ra khỏi bức ảnh (tọa độ < 0 hoặc > H, > W)
        // Đoạn code này tạo ra hiệu ứng phản chiếu (reflection). Hãy tưởng tượng bạn đặt một chiếc gương ở sát mép ảnh. 
        // Nếu tọa độ yêu cầu đọc điểm ảnh nằm ngoài rìa 2 pixel, nó sẽ lấy điểm ảnh nằm bên trong rìa 2 pixel để thay thế.
        int p_h = 2 * H;
        int gy_ref = gy % p_h;
        if (gy_ref < 0) gy_ref += p_h;
        if (gy_ref >= H) gy_ref = p_h - 1 - gy_ref;
        
        int p_w = 2 * W;
        int gx_ref = gx % p_w;
        if (gx_ref < 0) gx_ref += p_w;
        if (gx_ref >= W) gx_ref = p_w - 1 - gx_ref;
        
        // Sau khi đã tính toán xong tọa độ an toàn (gy_ref, gx_ref), 
        // lệnh này chính thức thực hiện việc chép dữ liệu từ RAM của GPU (d_input) vào bộ nhớ siêu tốc của Block (sm_input) để chuẩn bị cho bước tính toán ở phía sau.
        sm_input[r_y][r_x] = d_input[gy_ref * W + gx_ref];
    }
    // Buộc tất cả các threads chờ nhau tải xong mới làm tiếp
    __syncthreads();
    
    // VÒNG LẶP 2: Nhân chập (Convolution) theo chiều NGANG
    int total_row_elements = tile_size * 16;
    for (int i = tid; i < total_row_elements; i += 256) {
        int r_y = i / 16;
        int r_x = i % 16;
        float acc = 0.0f;
        for (int k = -radius; k <= radius; k++) {
            acc += sm_input[r_y][r_x + radius + k] * d_kernel[k + radius];
        }
        sm_row[r_y][r_x] = acc;
    }
    __syncthreads();
    
    // VÒNG LẶP 3: Nhân chập theo chiều DỌC và xuất ra biến d_output
    if (tx < 16 && ty < 16) {
        int gy = y0 + ty;
        int gx = x0 + tx;
        if (gx < W && gy < H) {
            float acc = 0.0f;
            for (int k = -radius; k <= radius; k++) {
                acc += sm_row[ty + radius + k][tx] * d_kernel[k + radius];
            }
            d_output[gy * W + gx] = acc;
        }
    }
}

// Kernel giảm kích thước ảnh đi một nửa (Lấy mẫu xuống / Subsampling)
extern "C" __global__
void downsample_kernel(const float* d_input, float* d_output, int H_in, int W_in, int H_out, int W_out) 
{
    // Ví dụ dễ hiểu: blockIdx là số thứ tự của xe buýt, blockDim là số ghế trên mỗi xe, threadIdx là số ghế của bạn. 
    // Nhân và cộng lại sẽ ra số thứ tự chính xác của bạn trong toàn bộ hành khách.
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    // Ánh xạ 1 pixel đầu ra với pixel gốc tại vị trí nhân đôi (x*2, y*2)
    if (x < W_out && y < H_out) {
        d_output[y * W_out + x] = d_input[(y * 2) * W_in + (x * 2)]; // thực hiện nhảy cóc 1 pixel, 
        // pixel (1,1) lấy màu của pixel (2,2)
    }
}

// ---------------------------------------------------------
// 2. KERNEL CHO DESCRIPTOR GENERATION
// Mục đích: Xây dựng vector đặc trưng 128 chiều cho từng điểm Keypoint.
// ---------------------------------------------------------
extern "C" __global__
void compute_sift_descriptors(
    const float* d_img, const float* d_kp_x, const float* d_kp_y, 
    const float* d_kp_scale, const float* d_kp_angle, float* d_descriptors, 
    int H, int W, int num_keypoints) 
{
    // Mỗi Block GPU đảm nhận xử lý 1 Keypoint duy nhất
    int kp_idx = blockIdx.x;
    if (kp_idx >= num_keypoints) return;
    
    // Khởi tạo vùng không gian 16x16 pixel xung quanh keypoint
    int tid = threadIdx.x;
    int tx = (tid % 16) - 8;
    int ty = (tid / 16) - 8;
    
    // Tải thông số tọa độ, tỷ lệ và góc quay của điểm Keypoint đó
    float kpt_x = d_kp_x[kp_idx];
    float kpt_y = d_kp_y[kp_idx];
    float kpt_scale = d_kp_scale[kp_idx];
    float kpt_angle = d_kp_angle[kp_idx] * 3.14159265f / 180.0f; 
    
    // Bộ đệm Histogram 128 chiều trên Shared Memory (16 vùng x 8 hướng)
    __shared__ float sm_hist[128];
    if (tid < 128) sm_hist[tid] = 0.0f;
    __syncthreads();
    
    // Tính toán góc xoay (để ảnh không bị ảnh hưởng khi xoay vật thể)
    float hist_width = 3.0f * kpt_scale; 
    float cos_t = cosf(kpt_angle);
    float sin_t = sinf(kpt_angle);
    
    float rx = (tx + 0.5f) * hist_width / 4.0f; 
    float ry = (ty + 0.5f) * hist_width / 4.0f;
    
    float rot_x = rx * cos_t - ry * sin_t;
    float rot_y = rx * sin_t + ry * cos_t;
    
    int img_x = (int)roundf(kpt_x + rot_x);
    int img_y = (int)roundf(kpt_y + rot_y);
    
    // Tính Gradient (độ lớn 'mag' và hướng 'ori' của viền)
    if (img_x > 0 && img_x < W - 1 && img_y > 0 && img_y < H - 1) {
        float dx = d_img[img_y * W + (img_x + 1)] - d_img[img_y * W + (img_x - 1)];
        float dy = d_img[(img_y + 1) * W + img_x] - d_img[(img_y - 1) * W + img_x];
        // độ lớn thay đổi
        float mag = sqrtf(dx * dx + dy * dy);
        // hướng thay đổi
        float ori = atan2f(dy, dx);
        
        // Gán trọng số theo chuông Gaussian (Càng xa tâm, giá trị càng giảm)
        float weight = expf(-(rx * rx + ry * ry) / (2.0f * (hist_width/2.0f)*(hist_width/2.0f)));
        mag *= weight;
        
        // Điều chỉnh hướng theo hướng gốc của Keypoint
        ori -= kpt_angle;
        while (ori < 0.0f) ori += 2.0f * 3.14159265f;
        while (ori >= 2.0f * 3.14159265f) ori -= 2.0f * 3.14159265f;
        
        float x_bin = (tx + 8.0f) / 4.0f; 
        float y_bin = (ty + 8.0f) / 4.0f; 
        float o_bin = ori * 8.0f / (2.0f * 3.14159265f); 
        
        int x_idx = (int)floorf(x_bin);
        int y_idx = (int)floorf(y_bin);
        int o_idx = (int)floorf(o_bin) % 8;
        
        // Nạp gradient vào biểu đồ Histogram (Dùng atomicAdd để không bị xung đột bộ nhớ giữa các threads)
        // 16 ô vuông, Bên trong mỗi ô vuông đó, các gradient sẽ được phân loại vào 8 hướng (ví dụ: lên, xuống, trái, phải, các đường chéo)
        // Tổng cộng ta có 16 * 8 = 128 chiều
        if (x_idx >= 0 && x_idx < 4 && y_idx >= 0 && y_idx < 4) {
            int hist_idx = (y_idx * 4 + x_idx) * 8 + o_idx;
            atomicAdd(&sm_hist[hist_idx], mag);
        }
    }
    __syncthreads();
    
    // ==============================================
    // 3 BƯỚC CHUẨN HÓA (Để tránh lỗi thay đổi độ sáng)
    // ==============================================
    
    // Bước chuẩn hóa 1: Tính tổng bình phương
    __shared__ float sum_sq;
    if (tid == 0) sum_sq = 0.0f;
    __syncthreads();
    
    if (tid < 128) {
        float val = sm_hist[tid];
        atomicAdd(&sum_sq, val * val);
    }
    __syncthreads();
    
    // Bước chuẩn hóa 2: Chia cho tổng và giới hạn (Cap) giá trị ở mức tối đa 0.2
    if (tid < 128) {
        float val = sm_hist[tid];
        float norm = sqrtf(sum_sq) + 1e-7f;
        val /= norm;
        if (val > 0.2f) val = 0.2f;
        sm_hist[tid] = val;
    }
    __syncthreads();
    
    // Bước chuẩn hóa 3: Tính tổng lại 1 lần nữa sau khi đã giới hạn 0.2
    if (tid == 0) sum_sq = 0.0f;
    __syncthreads();
    
    if (tid < 128) {
        float val = sm_hist[tid];
        atomicAdd(&sum_sq, val * val);
    }
    __syncthreads();
    
    // Xuất giá trị Descriptor 128 chiều cuối cùng ra biến trả về
    if (tid < 128) {
        float val = sm_hist[tid];
        float norm = sqrtf(sum_sq) + 1e-7f;
        val /= norm;
        d_descriptors[kp_idx * 128 + tid] = val;
    }
}
'''


## 3. Biên dịch kernel bằng CuPy

`cp.RawKernel(cuda_code, "<tên hàm>")` sẽ biên dịch (JIT — Just-In-Time) đoạn C++ ở trên thành mã máy GPU và trả về một đối tượng Python có thể gọi trực tiếp như hàm. Cờ `--use_fast_math` đánh đổi một chút độ chính xác số học để lấy tốc độ tính sin/cos/sqrt nhanh hơn.


In [11]:
# Biên dịch chuỗi C++ thành mã máy chạy trên GPU (Sử dụng --use_fast_math để tăng tốc tính toán lượng giác)
if CUPY_AVAILABLE:
    fused_kernel = cp.RawKernel(cuda_code, 'fused_gaussian_kernel', options=("--use_fast_math",))
    downsample_kernel = cp.RawKernel(cuda_code, 'downsample_kernel', options=("--use_fast_math",))
    descriptor_kernel = cp.RawKernel(cuda_code, 'compute_sift_descriptors', options=("--use_fast_math",))


## 4. Các hàm Python hỗ trợ

### 4.1. `make_gaussian_kernel_1d`
Tạo mảng trọng số Gauss 1 chiều trên **CPU** (dùng NumPy) — đây là bước chuẩn bị nhẹ, chạy 1 lần, không cần GPU. Bán kính lọc (`radius`) được tính theo quy tắc "cắt ở 4 lần sigma" (`truncate=4.0`), là ngưỡng phổ biến để bỏ qua phần đuôi rất nhỏ của phân bố Gauss mà không ảnh hưởng đáng kể tới kết quả.


In [12]:
# Hàm Python chuẩn bị mảng trọng số ma trận cho Kernel (Chạy trên CPU 1 lần ban đầu)
def make_gaussian_kernel_1d(sigma, truncate=4.0):
    radius = int(truncate * float(sigma) + 0.5)
    x = np.arange(-radius, radius + 1, dtype=np.float32)
    k = np.exp(-(x * x) / (2.0 * sigma * sigma))
    k /= k.sum() # Cân bằng trọng số để tổng = 1
    return k.astype(np.float32)


### 4.2. `build_gaussian_pyramid_with_lib` — phiên bản CPU (dùng để đối chiếu benchmark)

Dùng thẳng các hàm có sẵn của OpenCV (`cv2.GaussianBlur`, `cv2.resize`) để xây Gaussian Pyramid theo đúng công thức chuẩn SIFT: mỗi octave có `num_scales` ảnh mờ dần với `sigma = sigma_base * k^s`, trong đó `k = 2^(1/num_scales)`. Sau mỗi octave, ảnh ở giữa dải (`num_scales // 2`) được thu nhỏ 1 nửa để làm ảnh gốc cho octave kế tiếp.


In [13]:
# =============================================================================
# HÀM THỰC THI GAUSSIAN PYRAMID TRÊN CPU (DÙNG ĐỂ BENCHMARK)
# Bổ sung hàm này để mã có thể chạy mà không báo lỗi NameError.
# =============================================================================
def build_gaussian_pyramid_with_lib(image, num_octaves=4, num_scales=5, sigma_base=1.6):
    k = 2.0 ** (1.0 / num_scales)
    current_img = image
    pyramid = []
    
    for octave in range(num_octaves):
        octave_imgs = []
        for s in range(num_scales):
            sigma = sigma_base * (k ** s)
            # Hàm CPU của OpenCV áp dụng làm mờ
            blurred = cv2.GaussianBlur(current_img, (0, 0), sigmaX=sigma, sigmaY=sigma)
            octave_imgs.append(blurred)
        pyramid.append(octave_imgs)
        # Downsample cho dải Octave tiếp theo
        current_img = cv2.resize(octave_imgs[num_scales // 2], (0, 0), fx=0.5, fy=0.5, interpolation=cv2.INTER_NEAREST)
    return pyramid


### 4.3. `run_gaussian_pyramid_gpu` — phiên bản GPU

Đây là bản tăng tốc của bước trên bằng GPU:

1. Đẩy ảnh gốc từ CPU (NumPy) sang GPU (`cp.asarray`).
2. Tính trước các kernel Gauss 1D và bán kính cho từng scale (chạy 1 lần trên CPU, rẻ).
3. Tạo **CUDA Streams** — mỗi scale trong 1 octave chạy trên 1 stream riêng, cho phép GPU xử lý **song song** nhiều mức làm mờ khác nhau cùng lúc, thay vì tuần tự từng cái một.
4. Với mỗi octave: gọi `fused_kernel` cho từng scale (trên các stream khác nhau), đồng bộ hoá (`stream.synchronize()`), rồi gọi `downsample_kernel` trên ảnh ở giữa dải để tạo ảnh gốc cho octave tiếp theo.

So với hàm CPU, cấu trúc song song hoá theo cả 2 chiều: song song bên trong 1 lần Gaussian blur (hàng nghìn threads/pixel) **và** song song giữa các scale (multi-stream).


In [14]:
# =============================================================================
# HÀM THỰC THI GAUSSIAN PYRAMID TRÊN GPU
# =============================================================================
def run_gaussian_pyramid_gpu(image, num_octaves=4, num_scales=5, sigma_base=1.6):
    # Đẩy ảnh từ CPU sang RAM của GPU
    d_current = cp.asarray(image, dtype=cp.float32)
    k = 2.0 ** (1.0 / num_scales)
    
    # Tính toán trước các kernel bộ lọc để tiết kiệm thời gian
    dev_kernels = []
    radii = []
    for s in range(num_scales):
        sigma = sigma_base * (k ** s)
        hk = make_gaussian_kernel_1d(sigma, truncate=4.0)
        radii.append(hk.shape[0] // 2)
        dev_kernels.append(cp.asarray(hk, dtype=cp.float32))
        
    # Tạo Stream CUDA để GPU xử lý các độ mờ (scales) SONG SONG thay vì tuần tự
    streams = [cp.cuda.Stream() for _ in range(num_scales)]
    mid_idx = num_scales // 2

    for _ in range(num_octaves):
        H, W = d_current.shape
        grid = (math.ceil(W / 16), math.ceil(H / 16), 1)
        block = (16, 16, 1)

        d_outs = [cp.empty((H, W), dtype=cp.float32) for _ in range(num_scales)]

        # Chạy kernel song song bằng việc gán vào các stream khác nhau
        for s in range(num_scales):
            with streams[s]:
                fused_kernel(grid, block, (d_current, d_outs[s], dev_kernels[s], np.int32(radii[s]), np.int32(H), np.int32(W)))

        # Chờ toàn bộ các stream chạy xong (Đồng bộ)
        for stream in streams:
            stream.synchronize()
            
        d_mid = d_outs[mid_idx]
        
        # Thiết lập kích thước cho lần downsample
        H_new = math.ceil(H / 2.0)
        W_new = math.ceil(W / 2.0)
        d_current_new = cp.empty((H_new, W_new), dtype=cp.float32)
        
        # Giảm kích thước ảnh và cập nhật để lặp Octave mới
        downsample_kernel((math.ceil(W_new / 16), math.ceil(H_new / 16), 1), block, 
                          (d_mid, d_current_new, np.int32(H), np.int32(W), np.int32(H_new), np.int32(W_new)))
        d_current = d_current_new


## 5. Hàm benchmark tổng hợp — `benchmark_hybrid_sift`

Hàm này so sánh trực tiếp **CPU (OpenCV)** với **GPU (CuPy kernel)** trên cả 2 bước, cho từng ảnh trong bộ dữ liệu **DIV2K** (thư mục `DIV2K_train_HR` trong Google Drive, lấy 10 ảnh PNG đầu tiên).

Quy trình cho mỗi ảnh:

1. **Đọc ảnh**, chuyển sang thang xám (grayscale), chuẩn hoá về khoảng [0, 1].
2. **Benchmark Gaussian Pyramid**: đo thời gian trung bình (`n_runs=3` lần) của `build_gaussian_pyramid_with_lib` (CPU) và `run_gaussian_pyramid_gpu` (GPU, có `cp.cuda.Stream.null.synchronize()` để đảm bảo đo đúng thời gian GPU đã chạy xong, tránh đo nhầm thời gian "gọi hàm không đồng bộ").
3. **Lấy Keypoints** bằng `cv2.SIFT_create().detect()` trên CPU — vì bước dò điểm đặc trưng không phải là nút thắt cổ chai, nên vẫn giữ nguyên trên CPU để đơn giản hoá.
4. **Chuẩn bị dữ liệu keypoint** (toạ độ x, y, scale, angle) và đẩy sang GPU.
5. **Warm-up** kernel descriptor 1 lần (để loại bỏ độ trễ biên dịch/khởi động JIT lần đầu ra khỏi phép đo).
6. **Benchmark Descriptor Generation**: so sánh `sift.compute()` (CPU) với `descriptor_kernel` (GPU).
7. Tổng hợp thời gian 2 chặng (pyramid + descriptor) và in ra tốc độ tăng tốc (speedup = thời gian CPU / thời gian GPU).

Cuối cùng in bảng tổng kết trung bình trên toàn bộ ảnh đã test.


In [ ]:
# =============================================================================
# BENCHMARK HYBRID SIFT
# =============================================================================
def benchmark_hybrid_sift():
    print("=" * 70)
    print("  V8: HYBRID SIFT BENCHMARK (Pyramid + Descriptor)")
    print("=" * 70)

    if not CV2_AVAILABLE or not CUPY_AVAILABLE:
        print("Cần OpenCV và CuPy để test.")
        return

    # Quét thư mục để lấy các tệp hình ảnh
    div2k_dir = os.path.join('/content/drive/MyDrive/Colab Notebooks', 'DIV2K_train_HR')
    image_paths = sorted(glob.glob(os.path.join(div2k_dir, "*.png")))[:10]

    if not image_paths:
        print("Không có ảnh PNG nào để test. Bạn vui lòng kiểm tra lại đường dẫn thư mục.")
        return

    sift = cv2.SIFT_create()
    all_cpu_times = []
    all_gpu_times = []
    n_runs = 3 # Số lần chạy để lấy mức trung bình (càng cao càng chính xác)

    print(f"Tiến hành Benchmark {len(image_paths)} ảnh...\n")
    print(f"(Thời gian GPU chỉ tính thời gian tính toán ma trận thực thi kernel)\n")

    for i, img_path in enumerate(image_paths, 1):
        # Đọc và chuẩn hóa ảnh về định dạng 0-1
        img_color = cv2.imread(img_path)
        if img_color is None: continue
        gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
        image_norm = gray.astype(np.float32) / 255.0
        
        H, W = gray.shape

        # === 1. BENCHMARK PHẦN 1: GAUSSIAN PYRAMID ===
        # Test thời gian CPU
        cpu_pyr_times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _ = build_gaussian_pyramid_with_lib(image_norm, num_octaves=4, num_scales=5, sigma_base=1.6)
            cpu_pyr_times.append(time.perf_counter() - t0)
        avg_cpu_pyr = 1000.0 * np.mean(cpu_pyr_times)

				# WARMUP: Chạy mồi Gaussian Pyramid 1 lần để GPU biên dịch xong xuôi
        run_gaussian_pyramid_gpu(image_norm)
        cp.cuda.Stream.null.synchronize()
        
        # Test thời gian GPU
        gpu_pyr_times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            run_gaussian_pyramid_gpu(image_norm)
            cp.cuda.Stream.null.synchronize() # Ép buộc chờ GPU chạy xong mới chốt thời gian
            gpu_pyr_times.append(time.perf_counter() - t0)
        avg_gpu_pyr = 1000.0 * np.mean(gpu_pyr_times)


        # === 2. LẤY KEYPOINTS ĐỂ CHUẨN BỊ CHO BƯỚC DESCRIPTOR ===
        # Dùng OpenCV trên CPU để sinh điểm (Vì đây không phải nút thắt cổ chai hiệu suất)
        kp_cpu = sift.detect(gray, None)
        num_kp = len(kp_cpu)
        if num_kp == 0: continue

        # Bóc tách tọa độ từ Keypoint của OpenCV
        arr_x = np.array([k.pt[0] for k in kp_cpu], dtype=np.float32)
        arr_y = np.array([k.pt[1] for k in kp_cpu], dtype=np.float32)
        arr_scale = np.array([k.size / 2.0 for k in kp_cpu], dtype=np.float32)
        arr_angle = np.array([k.angle for k in kp_cpu], dtype=np.float32)
        
        # Gửi sang GPU
        d_x = cp.asarray(arr_x)
        d_y = cp.asarray(arr_y)
        d_scale = cp.asarray(arr_scale)
        d_angle = cp.asarray(arr_angle)
        d_img_base = cp.asarray(gray, dtype=cp.float32)
        d_desc = cp.empty((num_kp, 128), dtype=cp.float32)

        # Cấu hình mỗi Keypoints chạy trên 1 Block (Mỗi block có 256 threads)
        grid_desc = (num_kp, 1, 1)
        block_desc = (256, 1, 1)
        
        # WARMUP: Chạy mồi Descriptor để GPU biên dịch lần đầu (tránh làm sai lệch Benchmark)
        descriptor_kernel(grid_desc, block_desc, (d_img_base, d_x, d_y, d_scale, d_angle, d_desc, np.int32(H), np.int32(W), np.int32(num_kp)))
        cp.cuda.Stream.null.synchronize()

        # === 3. BENCHMARK PHẦN 2: DESCRIPTOR GENERATION ===
        # Test CPU 
        cpu_desc_times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _, des_cpu = sift.compute(gray, kp_cpu)
            cpu_desc_times.append(time.perf_counter() - t0)
        avg_cpu_desc = 1000.0 * np.mean(cpu_desc_times)
        
        # Test GPU
        gpu_desc_times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            descriptor_kernel(grid_desc, block_desc, (d_img_base, d_x, d_y, d_scale, d_angle, d_desc, np.int32(H), np.int32(W), np.int32(num_kp)))
            cp.cuda.Stream.null.synchronize()
            gpu_desc_times.append(time.perf_counter() - t0)
        avg_gpu_desc = 1000.0 * np.mean(gpu_desc_times)

        # Tổng hợp thời gian 2 chặng
        avg_cpu_total = avg_cpu_pyr + avg_cpu_desc
        avg_gpu_total = avg_gpu_pyr + avg_gpu_desc
        
        all_cpu_times.append(avg_cpu_total)
        all_gpu_times.append(avg_gpu_total)

        # In kết quả chi tiết từng ảnh
        print(f"[{i}/{len(image_paths)}] {os.path.basename(img_path)} ({W}x{H}) | Keypoints: {num_kp}")
        print(f"      -> CPU (Pyr + Desc)      : {avg_cpu_total:.1f} ms (Pyr: {avg_cpu_pyr:.1f}ms, Desc: {avg_cpu_desc:.1f}ms)")
        print(f"      -> GPU (Pyr + Desc)      : {avg_gpu_total:.2f} ms (Pyr: {avg_gpu_pyr:.2f}ms, Desc: {avg_gpu_desc:.2f}ms) | Speedup: {avg_cpu_total/avg_gpu_total:.2f}x\n")

    # Đánh giá chung
    if all_gpu_times:
        overall_cpu = float(np.mean(all_cpu_times))
        overall_gpu = float(np.mean(all_gpu_times))
        print("-" * 70)
        print(f"Tổng hợp CPU (2 Phần)  : {overall_cpu:.1f} ms/image")
        print(f"Tổng hợp GPU (2 Phần)  : {overall_gpu:.2f} ms/image")
        print(f"Hiệu suất Speedup chung : {overall_cpu / overall_gpu:.2f}x")
        print("-" * 70)


## 6. Chạy Benchmark

Chạy cell dưới đây để thực thi toàn bộ quá trình so sánh CPU vs GPU trên bộ ảnh DIV2K.

**Lưu ý trước khi chạy:**
- Cần chạy trên môi trường có **GPU Nvidia** (ví dụ Colab với Runtime → Change runtime type → GPU).
- Cần cài `cupy` (phiên bản khớp với CUDA Toolkit của máy, ví dụ `cupy-cuda12x`).
- Cần có thư mục `DIV2K_train_HR` chứa ảnh `.png` trong Google Drive tại đường dẫn `/content/drive/MyDrive/Colab Notebooks/DIV2K_train_HR` — hoặc bạn có thể sửa lại biến `div2k_dir` trong hàm để trỏ tới bộ ảnh của riêng bạn.


In [16]:
if __name__ == "__main__":
    benchmark_hybrid_sift()


  V8: HYBRID SIFT BENCHMARK (Pyramid + Descriptor)
Tiến hành Benchmark 10 ảnh...

(Thời gian GPU chỉ tính thời gian tính toán ma trận thực thi kernel)

[1/10] 0001.png (2040x1404) | Keypoints: 36861
      -> CPU (Pyr + Desc)      : 1779.4 ms (Pyr: 62.6ms, Desc: 1716.8ms)
      -> GPU (Pyr + Desc)      : 17.88 ms (Pyr: 10.32ms, Desc: 7.56ms) | Speedup: 99.52x

[2/10] 0002.png (2040x1848) | Keypoints: 17559
      -> CPU (Pyr + Desc)      : 1646.0 ms (Pyr: 78.9ms, Desc: 1567.1ms)
      -> GPU (Pyr + Desc)      : 17.69 ms (Pyr: 13.85ms, Desc: 3.84ms) | Speedup: 93.05x

[3/10] 0003.png (2040x1356) | Keypoints: 28473
      -> CPU (Pyr + Desc)      : 1416.1 ms (Pyr: 100.1ms, Desc: 1316.1ms)
      -> GPU (Pyr + Desc)      : 17.01 ms (Pyr: 11.16ms, Desc: 5.85ms) | Speedup: 83.24x

[4/10] 0004.png (2040x1344) | Keypoints: 6375
      -> CPU (Pyr + Desc)      : 651.5 ms (Pyr: 62.0ms, Desc: 589.5ms)
      -> GPU (Pyr + Desc)      : 11.35 ms (Pyr: 9.95ms, Desc: 1.40ms) | Speedup: 57.42x

[5/10] 0005